In [9]:

# MUTUAL FUND PERFORMANCE ANALYTICS
# ==========================================

import pandas as pd
import numpy as np
import scipy.stats as stats


ModuleNotFoundError: No module named 'pandas'

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

import os
import warnings

warnings.filterwarnings("ignore")

print("All libraries imported successfully!")
# Project directories

DATA_DIR = "data"
OUTPUT_DIR = "outputs"

# Create output directory if it does not exist

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Display settings

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.4f}".format)

# Risk-free rate
RISK_FREE_RATE = 0.065

# Trading days
TRADING_DAYS = 252

print("Project initialized successfully!")

# Load mutual fund NAV data

fund_nav = pd.read_csv(
    os.path.join(DATA_DIR, "fund_nav.csv")
)

# Load Nifty 50 data

nifty50 = pd.read_csv(
    os.path.join(DATA_DIR, "nifty50.csv")
)

# Load Nifty 100 data

nifty100 = pd.read_csv(
    os.path.join(DATA_DIR, "nifty100.csv")
)

# Load mutual fund details

fund_details = pd.read_csv(
    os.path.join(DATA_DIR, "fund_details.csv")
)

print("All datasets loaded successfully!") 

print("FUND NAV DATA")
display(fund_nav.head())

print("NIFTY 50 DATA")
display(nifty50.head())

print("NIFTY 100 DATA")
display(nifty100.head())

print("FUND DETAILS")
display(fund_details.head()) 

# Convert dates

fund_nav["Date"] = pd.to_datetime(
    fund_nav["Date"],
    errors="coerce"
)

nifty50["Date"] = pd.to_datetime(
    nifty50["Date"],
    errors="coerce"
)

nifty100["Date"] = pd.to_datetime(
    nifty100["Date"],
    errors="coerce"
)

# Convert numeric values

fund_nav["NAV"] = pd.to_numeric(
    fund_nav["NAV"],
    errors="coerce"
)

nifty50["Close"] = pd.to_numeric(
    nifty50["Close"],
    errors="coerce"
)

nifty100["Close"] = pd.to_numeric(
    nifty100["Close"],
    errors="coerce"
)

fund_details["Expense_Ratio"] = pd.to_numeric(
    fund_details["Expense_Ratio"],
    errors="coerce"
)

print("Data type conversion completed!") 


# Check missing values

print("Missing values in Fund NAV:")
print(fund_nav.isnull().sum())

print("\nMissing values in Nifty 50:")
print(nifty50.isnull().sum())

print("\nMissing values in Nifty 100:")
print(nifty100.isnull().sum())

print("\nMissing values in Fund Details:")
print(fund_details.isnull().sum()) 

# Remove invalid records

fund_nav = fund_nav.dropna(
    subset=["Date", "Scheme_Name", "NAV"]
)

fund_nav = fund_nav[fund_nav["NAV"] > 0]

# Remove duplicate fund/date observations

fund_nav = fund_nav.drop_duplicates(
    subset=["Date", "Scheme_Name"]
)

# Sort NAV data

fund_nav = fund_nav.sort_values(
    ["Scheme_Name", "Date"]
).reset_index(drop=True)

# Clean benchmark data

nifty50 = nifty50.dropna(
    subset=["Date", "Close"]
).drop_duplicates(subset=["Date"])

nifty100 = nifty100.dropna(
    subset=["Date", "Close"]
).drop_duplicates(subset=["Date"])

nifty50 = nifty50[nifty50["Close"] > 0]
nifty100 = nifty100[nifty100["Close"] > 0]

print("Data cleaning completed!")
print("Number of schemes:", fund_nav["Scheme_Name"].nunique()) 


# Calculate daily returns for every scheme

fund_nav["Daily_Return"] = (
    fund_nav.groupby("Scheme_Name")["NAV"]
    .pct_change()
)

# Display sample results

display(
    fund_nav[
        ["Date", "Scheme_Name", "NAV", "Daily_Return"]
    ].head(15)
) 

# Summary statistics

return_summary = fund_nav.groupby(
    "Scheme_Name"
)["Daily_Return"].agg(
    ["mean", "std", "min", "max", "count"]
)

display(return_summary)

# Distribution plot

plt.figure(figsize=(12, 6))

sns.histplot(
    fund_nav["Daily_Return"].dropna(),
    bins=100,
    kde=True
)

plt.title("Distribution of Mutual Fund Daily Returns")
plt.xlabel("Daily Return")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()